# PubMed Document Retrieval & Cleaning Pipeline

---
## 📂 Step 1: Imports + Reproducibility

In [1]:
import os
import json
import math
import time
import random
import re
import unicodedata
import requests
import numpy as np
import torch
import ftfy
from tqdm import tqdm
from langdetect import detect, LangDetectException, DetectorFactory
from dotenv import load_dotenv

load_dotenv()

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
DetectorFactory.seed = 0

# NCBI API
NCBI_API_KEY = os.environ.get('NCBI_API_KEY', '0c02b6e7f8e56927b51f9421816475b02e08')
ENTREZ_EMAIL = os.environ.get('ENTREZ_EMAIL', 'kohwekiat@hotmail.com')  # ← fix
RATE_LIMIT = 0.11 if NCBI_API_KEY else 0.34
BATCH_SIZE = 20

DATA_DIR = os.environ.get('DATA_DIR', '/workspace/data')

print(f'SEED: {SEED}')
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'PyTorch version: {torch.__version__}')
print(f'NCBI API key: {"✅ found" if NCBI_API_KEY else "❌ not set"}')
print(f'Entrez email: {ENTREZ_EMAIL}')
print(f'Data dir: {DATA_DIR}')

SEED: 42
CUDA available: True
PyTorch version: 2.10.0+cu128
NCBI API key: ✅ found
Entrez email: kohwekiat@hotmail.com
Data dir: /workspace/data


---
## 📂 Step 2: Load Data

In [2]:

os.environ['DATA_DIR'] = '/workspace/data'
data_dir = os.environ.get('DATA_DIR', './data')
DATASET_REGISTRY = {}

# Only load these datasets
ALLOWED_DATASETS = {'13B1_golden', '13B2_golden', '13B3_golden', '13B4_golden', 'training13b'}

print('📂 Loading standard JSON files...')
if not os.path.isdir(data_dir):
    print(f'❌ Directory not found: {data_dir}')
else:
    for filename in sorted(os.listdir(data_dir)):
        if filename.endswith('.json'):
            dataset_name = os.path.splitext(filename)[0]

            if dataset_name not in ALLOWED_DATASETS:  # ← filter here
                print(f'⏭️  Skipping: {filename}')
                continue

            file_path = os.path.join(data_dir, filename)
            with open(file_path, 'r', encoding='utf-8') as f:
                try:
                    raw_data = json.load(f)
                    if isinstance(raw_data, dict) and 'questions' in raw_data:
                        data_list = raw_data['questions']
                    elif isinstance(raw_data, list):
                        data_list = raw_data
                    else:
                        data_list = [raw_data]
                    DATASET_REGISTRY[dataset_name] = {'data': data_list}
                except json.JSONDecodeError as e:
                    print(f'⚠️ Could not parse {filename}: {e}')

print('\n📋 Dataset Inventory:')
for name, meta in DATASET_REGISTRY.items():
    print(f'  [{name}] — {len(meta["data"])} documents')

📂 Loading standard JSON files...
⏭️  Skipping: test_final.json
⏭️  Skipping: training_final.json

📋 Dataset Inventory:
  [13B1_golden] — 85 documents
  [13B2_golden] — 85 documents
  [13B3_golden] — 85 documents
  [13B4_golden] — 85 documents
  [training13b] — 5389 documents


---
## 📂 Step 3: Cleaning Functions

In [3]:
import re
import ftfy
from langdetect import detect, LangDetectException
from tqdm import tqdm

def remove_html_tags(text: str) -> str:
    return re.sub(r'<[^>]+>', '', text)

def fix_encoding(text: str) -> str:
    return ftfy.fix_text(text)

def is_low_quality(text: str, min_words: int = 5, max_symbol_ratio: float = 0.3) -> bool:
    words = text.split()
    if len(words) < min_words:
        return True
    symbols = sum(1 for c in text if not c.isalnum() and not c.isspace())
    if symbols / max(len(text), 1) > max_symbol_ratio:
        return True
    return False

def detect_language(text: str, target_lang: str = 'en') -> bool:
    try:
        return detect(text) == target_lang
    except LangDetectException:
        return False

def remove_pii(text: str) -> str:
    text = re.sub(r'[\w.+-]+@[\w-]+\.[\w.]+', '[EMAIL]', text)
    text = re.sub(r'\b\d{3}[-.\s]?\d{3}[-.\s]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b\d{3}-\d{2}-\d{4}\b', '[SSN]', text)
    return text

def parse_pubmed_abstract(text: str) -> dict:
    """Extract title, authors and abstract from PubMed plain text."""
    
    lines = text.strip().splitlines()
    
    # Title — line after first blank line
    title = ''
    for i, line in enumerate(lines):
        if line.strip() == '' and i > 0:
            # Next non-empty line is title
            for j in range(i+1, len(lines)):
                if lines[j].strip():
                    title = lines[j].strip()
                    break
            break
    
    # Authors — line after title block (contains initials pattern)
    authors = ''
    author_match = re.search(r'^[A-Z][a-z]+\s+\w{1,3}(?:\(\d+\))?,', text, re.MULTILINE)
    if author_match:
        authors = author_match.group(0).strip()

    # Abstract — try header first, then after "Author information:" block
    abstract_match = re.search(
        r'(ABSTRACT|INTRODUCTION|BACKGROUND|OBJECTIVE|OBJECTIVES|SUMMARY|PURPOSE|AIM|AIMS)\s*:',
        text, flags=re.IGNORECASE
    )
    if abstract_match:
        abstract = text[abstract_match.start():]
    else:
        # No header — abstract starts after Author information block
        author_info_match = re.search(r'Author information:.*?\n\n', text, flags=re.DOTALL)
        if author_info_match:
            abstract = text[author_info_match.end():]
        else:
            # Fallback — skip first 4 blank-line-separated blocks
            blocks = re.split(r'\n\n+', text)
            abstract = '\n\n'.join(blocks[3:]) if len(blocks) > 3 else ''

    # Cut off at end markers
    end_markers = [
        r'\nDOI:',
        r'\nPMID:',
        r'\nSimilar articles',
        r'\nCited by',
        r'\nPublication types',
        r'\nMeSH terms',
    ]
    for marker in end_markers:
        match = re.search(marker, abstract, flags=re.IGNORECASE)
        if match:
            abstract = abstract[:match.start()]
            break

    return {
        'title': title,
        'authors': authors,
        'abstract': abstract.strip()
    }

def clean_text(text: str) -> str | None:
    """Full cleaning pipeline for fetched abstract text."""
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return text

def clean_cached_abstract(text: str) -> str | None:
    """Faster cleaning for PubMed abstracts — skips language detection."""
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    text = remove_pii(text)
    return text

def clean_document(doc: dict) -> dict | None:
    text = doc.get('body', '')
    if not text and 'snippets' in doc:
        text = ' '.join([s.get('text', '') for s in doc['snippets']])
    text = remove_html_tags(text)
    text = fix_encoding(text)
    text = text.strip()
    if is_low_quality(text):
        return None
    if not detect_language(text):
        return None
    text = remove_pii(text)
    return {**doc, 'text': text}

# Apply cleaning
cleaned_datasets = {}
for name, meta in DATASET_REGISTRY.items():
    print(f'🧹 Cleaning dataset: {name} ({len(meta["data"])} documents)')
    cleaned_docs = []
    for doc in tqdm(meta['data'], desc=f'Cleaning {name}'):
        cleaned_doc = clean_document(doc)
        if cleaned_doc:
            cleaned_docs.append(cleaned_doc)
    cleaned_datasets[name] = cleaned_docs
    print(f'✅ Cleaned {len(cleaned_docs)} documents from {name}')

🧹 Cleaning dataset: 13B1_golden (85 documents)


Cleaning 13B1_golden: 100% 85/85 [00:00<00:00, 142.53it/s]


✅ Cleaned 73 documents from 13B1_golden
🧹 Cleaning dataset: 13B2_golden (85 documents)


Cleaning 13B2_golden: 100% 85/85 [00:00<00:00, 375.02it/s]


✅ Cleaned 75 documents from 13B2_golden
🧹 Cleaning dataset: 13B3_golden (85 documents)


...

✅ Cleaned 76 documents from 13B3_golden
🧹 Cleaning dataset: 13B4_golden (85 documents)


Cleaning 13B4_golden: 100% 85/85 [00:00<00:00, 272.84it/s]


✅ Cleaned 70 documents from 13B4_golden
🧹 Cleaning dataset: training13b (5389 documents)


Cleaning training13b: 100% 5389/5389 [00:10<00:00, 495.39it/s] 

✅ Cleaned 4804 documents from training13b


In [4]:
def avg_word_length(text: str) -> float:
    words = text.split()
    return sum(len(w) for w in words) / max(len(words), 1)

def bullet_density(text: str) -> float:
    lines = text.splitlines()
    bullet_lines = sum(1 for l in lines if l.strip().startswith(('•', '-', '*', '·')))
    return bullet_lines / max(len(lines), 1)

def passes_quality_filter(doc: dict, verbose: bool = False) -> bool:
    text = doc.get('text', '')
    if not text or len(text) < 20:
        if verbose: print(f'FAIL length: {len(text)}')
        return False
    awl = avg_word_length(text)
    if not (2.5 <= awl <= 15):
        if verbose: print(f'FAIL awl: {awl:.2f}')
        return False
    if bullet_density(text) > 0.6:
        if verbose: print('FAIL bullets')
        return False
    return True

quality_filtered = {}
for name, docs in cleaned_datasets.items():
    before = len(docs)
    filtered = [doc for doc in docs if passes_quality_filter(doc)]
    quality_filtered[name] = filtered
    print(f'[{name}] quality filter: {before} → {len(filtered)} docs')

print('\n✅ Quality filter complete')


[13B1_golden] quality filter: 73 → 73 docs
[13B2_golden] quality filter: 75 → 75 docs
[13B3_golden] quality filter: 76 → 76 docs
[13B4_golden] quality filter: 70 → 70 docs
[training13b] quality filter: 4804 → 4803 docs

✅ Quality filter complete


In [5]:
import unicodedata

def normalize_text(text: str, form: str = 'NFC') -> str:
    text = unicodedata.normalize(form, text)
    text = re.sub(r'\r\n|\r', '\n', text)
    text = re.sub(r'\t', ' ', text)
    text = re.sub(r' {2,}', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)
    text = text.strip()
    return text

normalized_datasets = {}
for name, docs in quality_filtered.items():
    normalized = [{**doc, 'text': normalize_text(doc['text'])} for doc in docs]
    normalized_datasets[name] = normalized

training_raw_data = normalized_datasets['training13b']
test_raw_data = (
    normalized_datasets['13B1_golden'] +
    normalized_datasets['13B2_golden'] +
    normalized_datasets['13B3_golden'] +
    normalized_datasets['13B4_golden']
)

print('✅ Normalization complete')
print(f'Training docs: {len(training_raw_data)}')
print(f'Test docs:     {len(test_raw_data)}')
for name, docs in normalized_datasets.items():
    if docs:
        sample = docs[0]['text']
        print(f'\n[{name}] sample: {sample[:120]}...' if len(sample) > 120 else f'\n[{name}] sample: {sample}')

✅ Normalization complete
Training docs: 4803
Test docs:     294

[13B1_golden] sample: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subtype (CMS) groups?

[13B2_golden] sample: Which ensemble machine-learning framework has been developed harnessing UK biobank data?

[13B3_golden] sample: How many primary genetic associations were identified through pQTL mapping within the Pharma Proteomics Project?

[13B4_golden] sample: Should Zotiraciclib be used for glioblastoma?

[training13b] sample: Is Hirschsprung disease a mendelian or a multifactorial disorder?


---
## 📂 Step 4: Fetching documents

In [6]:
from Bio import Entrez
import json

BATCH_SIZE = 20
Entrez.email = os.environ.get('ENTREZ_EMAIL', 'your_email@example.com')
Entrez.api_key = NCBI_API_KEY

def extract_pmid(url: str) -> str:
    """Extract PMID from PubMed URL."""
    return url.rstrip('/').split('/')[-1]

def fetch_pubmed_batch(urls: list, api_key: str = None) -> dict:
    """Fetch multiple abstracts via Biopython XML parser. Returns {url: parsed_dict}."""
    try:
        url_to_pmid = {}
        for url in urls:
            pmid = extract_pmid(url)
            if pmid.isdigit():
                url_to_pmid[url] = pmid

        if not url_to_pmid:
            return {}

        pmids = list(url_to_pmid.values())
        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(pmids),
            rettype="abstract",
            retmode="xml"
        )
        data = Entrez.read(handle)
        handle.close()

        pmid_to_parsed = {}
        for article in data["PubmedArticle"]:
            citation = article["MedlineCitation"]
            art = citation["Article"]
            pmid = str(citation["PMID"])

            title = str(art.get("ArticleTitle", ""))

            authors = []
            for a in art.get("AuthorList", []):
                if "LastName" in a:
                    authors.append(f"{a.get('ForeName', '')} {a.get('LastName', '')}".strip())

            abstract_sections = []
            if "Abstract" in art:
                for i, sec in enumerate(art["Abstract"]["AbstractText"], start=1):
                    label = sec.attributes.get("Label", "") if hasattr(sec, 'attributes') else ""
                    abstract_sections.append({
                        "section_id": i,
                        "label": label,
                        "text": str(sec)
                    })

            pmid_to_parsed[pmid] = {
                "pmid": pmid,
                "title": title,
                "authors": "; ".join(authors),
                "abstract_sections": abstract_sections,
                "abstract": " ".join(s["text"] for s in abstract_sections)
            }

        return {
            url: pmid_to_parsed.get(pmid, {})
            for url, pmid in url_to_pmid.items()
        }

    except Exception as e:
        print(f'Batch fetch error: {e}')
        return {}

# Test with first 3 URLs
test_urls = [s.get('document', '') for s in training_raw_data[0].get('snippets', [])[:3]]
print(f'Testing batch fetch with {len(test_urls)} URLs...')
results = fetch_pubmed_batch(test_urls, NCBI_API_KEY)
for url, parsed in results.items():
    print(f'\nURL: {url}')
    print(f'Title: {parsed.get("title", "")}')
    print(f'Authors: {parsed.get("authors", "")}')
    print(f'Abstract: {parsed.get("abstract", "")[:200]}...')

Testing batch fetch with 3 URLs...

URL: http://www.ncbi.nlm.nih.gov/pubmed/15829955
Title: A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease risk.
Authors: Eileen Sproat Emison; Andrew S McCallion; Carl S Kashuk; Richard T Bush; Elizabeth Grice; Shin Lin; Matthew E Portnoy; David J Cutler; Eric D Green; Aravinda Chakravarti
Abstract: The identification of common variants that contribute to the genesis of human inherited disorders remains a significant challenge. Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disord...

URL: http://www.ncbi.nlm.nih.gov/pubmed/15617541
Title: Studying the genetics of Hirschsprung's disease: unraveling an oligogenic disorder.
Authors: A S Brooks; B A Oostra; R M W Hofstra
Abstract: Hirschsprung's disease is characterized by the absence of ganglion cells in the myenteric and submucosal plexuses of the gastrointestinal tract. Genetic dissection was successful as nine genes and fou...

URL: http://www.ncbi.nlm.

In [7]:
# Deduplicate URLs (avoid redundant fetches)
# Collect all unique PubMed URLs across all docs
all_urls = set()
for doc in training_raw_data:
    for snippet in doc.get('snippets', []):
        url = snippet.get('document', '')
        if url:
            all_urls.add(url)

print(f'Total docs:        {len(training_raw_data)}')
print(f'Unique PubMed URLs: {len(all_urls)}')

# Estimate time
est_seconds = len(all_urls) * RATE_LIMIT
est_minutes = est_seconds / 60
print(f'\nEstimated fetch time: {est_minutes:.1f} minutes')
print(f'(Based on {RATE_LIMIT}s/request × {len(all_urls)} unique URLs)')

Total docs:        4803
Unique PubMed URLs: 37596

Estimated fetch time: 68.9 minutes
(Based on 0.11s/request × 37596 unique URLs)


In [8]:
# Load Cache (skip re-fetching on reruns) ──────────────────
cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')

if os.path.exists(cache_path):
    with open(cache_path, 'r', encoding='utf-8') as f:
        abstract_cache = json.load(f)
    print(f'✅ Loaded {len(abstract_cache)} cached abstracts')
else:
    print('❌ No cache found — run next cell first')

❌ No cache found — run next cell first


In [9]:
abstract_cache = {}
failed_urls = []
all_url_list = list(all_urls)
n_batches = math.ceil(len(all_url_list) / BATCH_SIZE)

for i in tqdm(range(0, len(all_url_list), BATCH_SIZE), desc='Fetching batches', total=n_batches):
    batch = all_url_list[i:i + BATCH_SIZE]
    results = fetch_pubmed_batch(batch, NCBI_API_KEY)

    for url in batch:
        if url in results and results[url]:
            abstract_cache[url] = results[url]  # ← dict now, not string
        else:
            failed_urls.append(url)

    time.sleep(RATE_LIMIT)

cache_path = os.path.join(DATA_DIR, 'pubmed_abstract_cache.json')
with open(cache_path, 'w', encoding='utf-8') as f:
    json.dump(abstract_cache, f, indent=2, ensure_ascii=False)
print(f'✅ Saved cache: {len(abstract_cache)} abstracts')

Fetching batches:  29% 540/1880 [18:50<3:44:25, 10.05s/it]

Batch fetch error: IncompleteRead(572 bytes read)


Fetching batches:  43% 817/1880 [28:07<30:38,  1.73s/it]  

Batch fetch error: Remote end closed connection without response


Fetching batches:  97% 1819/1880 [1:02:17<01:57,  1.92s/it]

Batch fetch error: IncompleteRead(941 bytes read)


Fetching batches: 100% 1880/1880 [1:04:20<00:00,  2.05s/it]


✅ Saved cache: 37483 abstracts


In [10]:
# Pre-process cache — already parsed by Biopython
print("Processing cache...")
cleaned_cache = {}
parsed_cache = {}

for url, parsed in tqdm(abstract_cache.items(), desc="Processing cache"):
    if not parsed or not parsed.get('abstract'):
        continue
    
    cleaned = clean_cached_abstract(parsed['abstract'])
    if cleaned:
        cleaned_cache[url] = cleaned
        parsed_cache[url] = {
            'title': parsed.get('title', ''),
            'authors': parsed.get('authors', ''),
            'abstract': cleaned,
            'abstract_sections': parsed.get('abstract_sections', [])
        }

print(f"✅ {len(cleaned_cache)} abstracts processed")

Processing cache...


Processing cache: 100% 37483/37483 [00:12<00:00, 3112.44it/s]

✅ 37475 abstracts processed


---
## 📂 Step 5: Build enriched training data

In [11]:
enriched_docs = []
skipped = 0

for doc in tqdm(training_raw_data, desc='Building enriched docs'):
    question = doc.get('body', '')
    snippets = doc.get('snippets', [])

    full_texts = []
    source_urls = []
    pmids = []
    seen_urls = set()

    for snippet in snippets:
        url = snippet.get('document', '')
        if url in seen_urls:
            continue
        seen_urls.add(url)

        if url in cleaned_cache:
            full_texts.append(cleaned_cache[url])
            source_urls.append(url)
            pmids.append(extract_pmid(url))
        else:
            fallback = clean_cached_abstract(snippet.get('text', ''))
            if fallback:
                full_texts.append(fallback)
                source_urls.append(url)
                pmids.append(extract_pmid(url))

    if not full_texts:
        skipped += 1
        continue

    for pmid, text, url in zip(pmids, full_texts, source_urls):
        parsed = parsed_cache.get(url, {})
        enriched_docs.append({
            **doc,
            "text": text,
            "title": parsed.get('title', ''),
            "authors": parsed.get('authors', ''),
            "question": question,
            "pmid": pmid,
            "source_url": url,
            "n_sources": len(source_urls)
        })

print(f'✅ Enriched docs: {len(enriched_docs)}')
print(f'⚠️  Skipped: {skipped}')

Building enriched docs: 100% 4803/4803 [00:00<00:00, 9372.40it/s] 

✅ Enriched docs: 39442
⚠️  Skipped: 0


---
## 📂 Step 6: Zero-shot Categorization on Full Abstracts

In [20]:
from transformers import pipeline
from tqdm import tqdm
from collections import Counter
import torch
import gc

CATEGORIES = [
    "Genetics & Mutations",
    "Cancer & Oncology",
    "Pharmacology & Drugs",
    "Neurology & Brain",
    "Cardiology & Heart",
    "Infectious Disease",
    "Immunology",
    "Metabolism & Diabetes",
    "Rare Diseases",
    "Surgery & Procedures",
    "Diagnostics & Imaging",
    "Mental Health",
    "Molecular Biology",
    "Clinical Guidelines",
    "Other"
]

device = 0 if torch.cuda.is_available() else -1
print(f'Using device: {"cuda" if device == 0 else "cpu"}')

print("Loading classifier...")
classifier = pipeline(
    "zero-shot-classification",
    model="cross-encoder/nli-deberta-v3-small",
    device=device
)
print("✅ Classifier ready!")

# Step 1: group by question id, join abstracts
question_groups = {}
for doc in enriched_docs:
    qid = doc.get('id', doc['question'])
    if qid not in question_groups:
        question_groups[qid] = {
            'texts': [],
            'docs': []
        }
    question_groups[qid]['texts'].append(doc['text'])
    question_groups[qid]['docs'].append(doc)

print(f'Total unique questions: {len(question_groups)}')

# Step 2: classify on joined text per question
for qid, group in tqdm(question_groups.items(), desc="Classifying"):
    joined_text = ' '.join(group['texts'])
    result = classifier(joined_text, CATEGORIES)
    category = result['labels'][0]
    confidence = round(result['scores'][0], 4)
    topic_id = CATEGORIES.index(category)

    # Step 3: copy category to all rows for this question
    for doc in group['docs']:
        doc['category'] = category
        doc['confidence'] = confidence
        doc['topic_id'] = topic_id

# Clear GPU memory
torch.cuda.empty_cache()
gc.collect()

cats = Counter(doc['category'] for doc in enriched_docs)
print(f'\n✅ Done! {len(cats)} categories')
print(f'{"Category":<40} {"Count":>6} {"%":>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / len(enriched_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

Using device: cuda
Loading classifier...


Loading weights: 100% 106/106 [00:00<00:00, 757.66it/s, Materializing param=pooler.dense.weight]                                     
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-small
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Classifier ready!
Total unique questions: 4803


Classifying: 100% 4803/4803 [20:38<00:00,  3.88it/s]



✅ Done! 15 categories
Category                                  Count      %
-------------------------------------------------------
Other                                     14717 (37.3%)
Molecular Biology                          4008 (10.2%)
Rare Diseases                              3205 (8.1%)
Immunology                                 3152 (8.0%)
Surgery & Procedures                       2161 (5.5%)
Diagnostics & Imaging                      2114 (5.4%)
Pharmacology & Drugs                       1749 (4.4%)
Genetics & Mutations                       1677 (4.3%)
Infectious Disease                         1547 (3.9%)
Cardiology & Heart                         1419 (3.6%)
Cancer & Oncology                          1226 (3.1%)
Neurology & Brain                          1053 (2.7%)
Clinical Guidelines                         709 (1.8%)
Metabolism & Diabetes                       474 (1.2%)
Mental Health                               231 (0.6%)


In [21]:
from collections import Counter

pmid_groups = {}
for doc in enriched_docs:
    pmid = doc['pmid']
    if pmid not in pmid_groups:
        pmid_groups[pmid] = {
            'pmid': pmid,
            'title': doc.get('title', ''),
            'authors': doc.get('authors', ''),
            'text': doc['text'],
            'source_url': doc['source_url'],
            'categories': []
        }
    pmid_groups[pmid]['categories'].append(doc['category'])

vector_db_docs = []
for pmid, data in pmid_groups.items():
    most_common_category = Counter(data['categories']).most_common(1)[0][0]
    vector_db_docs.append({
        'pmid': pmid,
        'title': data['title'],
        'authors': data['authors'],
        'text': data['text'],
        'source_url': data['source_url'],
        'category': most_common_category
    })

print(f'✅ Unique PMIDs for vector DB: {len(vector_db_docs)}')

✅ Unique PMIDs for vector DB: 37595


In [22]:
from collections import Counter

# Question level
cats = Counter(doc['category'] for doc in enriched_docs)
total = len(enriched_docs)
print(f'=== Question Level ({total} docs) ===')
print(f'{"Category":<40} {"Count":>6} {"%" :>6}')
print('-' * 55)
for cat, count in cats.most_common():
    pct = count / total * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

# Vector DB level
print(f'\n=== Vector DB Level ({len(vector_db_docs)} unique PMIDs) ===')
cats3 = Counter(doc['category'] for doc in vector_db_docs)
for cat, count in cats3.most_common():
    pct = count / len(vector_db_docs) * 100
    print(f'{cat:<40} {count:>6} ({pct:.1f}%)')

=== Question Level (39442 docs) ===
Category                                  Count      %
-------------------------------------------------------
Other                                     14717 (37.3%)
Molecular Biology                          4008 (10.2%)
Rare Diseases                              3205 (8.1%)
Immunology                                 3152 (8.0%)
Surgery & Procedures                       2161 (5.5%)
Diagnostics & Imaging                      2114 (5.4%)
Pharmacology & Drugs                       1749 (4.4%)
Genetics & Mutations                       1677 (4.3%)
Infectious Disease                         1547 (3.9%)
Cardiology & Heart                         1419 (3.6%)
Cancer & Oncology                          1226 (3.1%)
Neurology & Brain                          1053 (2.7%)
Clinical Guidelines                         709 (1.8%)
Metabolism & Diabetes                       474 (1.2%)
Mental Health                               231 (0.6%)

=== Vector DB Level (3759

---
## 📂 Step 7: Save final data in CSV and Json format

In [23]:
import pandas as pd

parsed_docs_dir = os.path.join(DATA_DIR, 'parsed_docs')
os.makedirs(parsed_docs_dir, exist_ok=True)

# Save as JSON
output_train = os.path.join(DATA_DIR, 'training_final.json')
with open(output_train, 'w', encoding='utf-8') as f:
    json.dump(enriched_docs, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(enriched_docs)} training docs to {output_train}')

# Save as CSV
csv_path = os.path.join(parsed_docs_dir, 'training_final.csv')
df = pd.DataFrame([{
    'question':   doc.get('question', ''),
    'type':       doc.get('type', ''),
    'title':      doc.get('title', ''),
    'authors':    doc.get('authors', ''),
    'text':       doc.get('text', ''),
    'category':   doc.get('category', ''),
    'topic_id':   doc.get('topic_id', ''),
    'confidence': doc.get('confidence', ''),
    'pmid':       doc.get('pmid', ''),
    'source_url': doc.get('source_url', ''),
    'n_sources':  doc.get('n_sources', 0),
} for doc in enriched_docs])  # ← enriched_docs
df.to_csv(csv_path, index=False, encoding='utf-8', quoting=1)
print(f'✅ Saved CSV to {csv_path}')
print(f'   Rows:    {len(df)}')

# Verify
with open(output_train, 'r', encoding='utf-8') as f:
    verify = json.load(f)
print(f'\n✅ Verified JSON: {len(verify)} docs')
print(f'   Fields:   {list(verify[0].keys())}')
print(f'   Category: {verify[0].get("category")}')
print(f'   Text:     {verify[0]["text"][:150]}...')

✅ Saved 39442 training docs to /workspace/data/training_final.json
✅ Saved CSV to /workspace/data/parsed_docs/training_final.csv
   Rows:    39442

✅ Verified JSON: 39442 docs
   Fields:   ['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets', 'text', 'title', 'authors', 'question', 'pmid', 'source_url', 'n_sources', 'category', 'confidence', 'topic_id']
   Category: Rare Diseases
   Text:     The identification of common variants that contribute to the genesis of human inherited disorders remains a significant challenge. Hirschsprung diseas...


In [24]:
# Build Test Docs — one row per question
test_docs = []

for doc in tqdm(test_raw_data, desc='Building test docs'):
    test_docs.append({
        'question':  doc.get('body', ''),
        'type':      doc.get('type', ''),
        'ideal_answer': doc.get('ideal_answer', ''),
        'documents': doc.get('documents', []),
        'snippets':  doc.get('snippets', []),
        'concepts':  doc.get('concepts', []),
        'id':        doc.get('id', '')
    })

print(f'✅ Test docs: {len(test_docs)}')

Building test docs: 100% 294/294 [00:00<00:00, 801251.06it/s]

✅ Test docs: 294


In [25]:
# Save test as JSON
output_test = os.path.join(DATA_DIR, 'test_final.json')
with open(output_test, 'w', encoding='utf-8') as f:
    json.dump(test_docs, f, indent=2, ensure_ascii=False)
print(f'✅ Saved {len(test_docs)} test docs to {output_test}')

# Save test as CSV
test_csv_path = os.path.join(parsed_docs_dir, 'test_final.csv')
df_test = pd.DataFrame([{
    'question':     doc.get('question', ''),
    'type':         doc.get('type', ''),
    'ideal_answer': doc.get('ideal_answer', ''),
    'id':           doc.get('id', ''),
} for doc in test_docs])
df_test.to_csv(test_csv_path, index=False, encoding='utf-8', quoting=1)
print(f'✅ Saved test CSV to {test_csv_path}')
print(f'   Rows:    {len(df_test)}')

# Verify
with open(output_test, 'r', encoding='utf-8') as f:
    verify_test = json.load(f)
print(f'\n✅ Verified JSON: {len(verify_test)} test docs')
print(f'   Fields:   {list(verify_test[0].keys())}')
print(f'   Question: {verify_test[0].get("question")[:100]}')
print(f'   Type:     {verify_test[0].get("type")}')

✅ Saved 294 test docs to /workspace/data/test_final.json
✅ Saved test CSV to /workspace/data/parsed_docs/test_final.csv
   Rows:    294

✅ Verified JSON: 294 test docs
   Fields:   ['question', 'type', 'ideal_answer', 'documents', 'snippets', 'concepts', 'id']
   Question: What proportion of colorectal cancer cases are not assignable to any of the consensus molecular subt
   Type:     factoid


In [26]:
import pandas as pd

pd.set_option('display.max_colwidth', 100)

df = pd.read_csv('/workspace/data/parsed_docs/training_final.csv')
print(f'Shape: {df.shape}')
df.head()

Shape: (39442, 11)


,question,type,title,authors,text,category,topic_id,confidence,pmid,source_url,n_sources
0,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,A common sex-dependent mutation in a RET enhancer underlies Hirschsprung disease risk.,Eileen Sproat Emison; Andrew S McCallion; Carl S Kashuk; Richard T Bush; Elizabeth Grice; Shin L...,The identification of common variants that contribute to the genesis of human inherited disorder...,Rare Diseases,8,0.103,15829955,http://www.ncbi.nlm.nih.gov/pubmed/15829955,9
1,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,Studying the genetics of Hirschsprung's disease: unraveling an oligogenic disorder.,A S Brooks; B A Oostra; R M W Hofstra,Hirschsprung's disease is characterized by the absence of ganglion cells in the myenteric and su...,Rare Diseases,8,0.103,15617541,http://www.ncbi.nlm.nih.gov/pubmed/15617541,9
2,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"Hirschsprung, RET-SOX and beyond: the challenge of examining non-mendelian traits (Review).",Carsten M Pusch; Maria M Sasiadek; Nikolaus Blin,"Hirschsprung disease (HSCR), or congenital intestinal aganglionosis, is a common hereditary diso...",Rare Diseases,8,0.103,12239580,http://www.ncbi.nlm.nih.gov/pubmed/12239580,9
3,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,Hirschsprung disease: etiologic implications of unsuccessful prenatal diagnosis.,A L Jarmas; D D Weaver; L M Padilla; E Stecker; H A Bender,We describe an infant with Hirschsprung disease (congenital aganglionosis of the intestine) invo...,Rare Diseases,8,0.103,6650562,http://www.ncbi.nlm.nih.gov/pubmed/6650562,9
4,Is Hirschsprung disease a mendelian or a multifactorial disorder?,summary,"Differential contributions of rare and common, coding and noncoding Ret mutations to multifactor...",Eileen Sproat Emison; Merce Garcia-Barcelo; Elizabeth A Grice; Francesca Lantieri; Jeanne Amiel;...,The major gene for Hirschsprung disease (HSCR) encodes the receptor tyrosine kinase RET. In a st...,Rare Diseases,8,0.103,20598273,http://www.ncbi.nlm.nih.gov/pubmed/20598273,9
